# Lab 08-02 — HotpotQA evidence graph: how gold paragraphs connect

**Track 08 · GraphRAG** — how a benchmark's own gold labels expose multi-hop structure.

Lab 01 built a graph from an LLM's triples. This lab builds the same kind of graph with **no model at all**: HotpotQA multi-hop questions ship with `supporting_facts` — the exact (paragraph, sentence) pairs a correct answer must combine. We turn those gold labels into a **co-evidence graph** — nodes are paragraphs that appear as gold evidence, and an edge between two paragraphs means some question demanded both. The result is a clean ground-truth view of what "multi-hop" structure actually looks like.

```text
hotpotqa dev set (gold supporting_facts)
  -> keep multi-hop questions (>= 2 gold paragraphs)
  -> co-evidence graph in pure networkx (edge weight = shared questions)
  -> print evidence chains + hub paragraphs
  -> verification gate (--verify)
```

The pipeline needs exactly four ingredients: the **hotpotqa dev set** for labels, **pure networkx** for the graph, and **no LLM, no embeddings** — nothing is generated, only read off the dataset.


## Setup

This notebook mirrors `src/curriculum/08-graphrag/02-evidence-graph.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

The headline: **no model needed — this is a ground-truth view of multi-hop structure.** Lab 01 paid for an LLM to mine triples out of text; here the graph comes straight from HotpotQA's gold `supporting_facts` labels, so there are no API calls, no embeddings, no randomness — every run produces the identical graph.

From the terminal, the lab runs as:

```bash
python src/curriculum/08-graphrag/02-evidence-graph.py          # run + demo
python src/curriculum/08-graphrag/02-evidence-graph.py --verify # verification gate
```

The next cell installs the lab-specific dependency (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependency (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   networkx -> the co-evidence graph (this lab's only dependency)
%pip install networkx


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# A notebook has no ``__file__``, so resolve the repo root by walking up
# from the kernel's working directory — this works whether the kernel
# launches from the repo root (like the lab script) or from the notebook's
# own folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import networkx as nx  # noqa: E402


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_QUESTIONS = 50` caps the pool at the first 50 multi-hop questions in the dev set — a deterministic head, so every run sees the same questions and the same graph. `MAX_CHAINS_SHOWN` and `TOP_HUBS` only control how much the demo prints; the graph itself always covers the full 50-question pool.


In [ ]:
# 1. Configuration
HOTPOT_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
N_QUESTIONS = 50  # pool cap: first N multi-hop questions (deterministic head)
MAX_CHAINS_SHOWN = 5  # how many example evidence chains to print
TOP_HUBS = 6  # how many highest-degree paragraphs to print


## 2. Load — dev set, keep only multi-hop questions (>= 2 gold paragraphs)

`supporting_facts` is a list of `(title, sentence_index)` pairs — the gold evidence a correct answer must combine. Multi-hop means the answer lives in the link *between* paragraphs, not inside one, so we keep only records whose gold evidence spans **>= 2 distinct paragraphs**, and stop once we have the first `n` of them. `evidence_chain` then sorts each question's facts into (title, sentence) order — the exact chain a retriever would need to surface.


In [ ]:
# 2. Load — dev set, keep only multi-hop questions (>= 2 gold paragraphs)
def load_multi_hop(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` questions whose gold evidence spans >= 2 paragraphs."""
    with open(path) as f:
        records = json.load(f)
    out: list[dict] = []
    for rec in records:
        gold_titles = {title for title, _ in rec["supporting_facts"]}
        if len(gold_titles) >= 2:
            out.append(rec)
        if len(out) >= n:
            break
    return out


def evidence_chain(rec: dict) -> list[tuple[str, int]]:
    """The (title, sentence_index) chain of a question's gold evidence."""
    facts = sorted(
        [(title, sent_idx) for title, sent_idx in rec["supporting_facts"]],
        key=lambda pair: (pair[0], pair[1]),
    )
    return facts


## 3. Experiment — co-evidence graph over the sampled questions

Every question whose gold evidence covers paragraphs A and B casts one vote for the pair (A, B). We collect those votes in a plain `networkx.Graph`: **nodes are paragraphs**, an edge exists when at least one question demanded both endpoints, and `weight` counts how many questions share the pair. Each node also remembers which question ids need it, so the graph stays traceable back to the data. The returned dict bundles the graph with the per-question evidence chains, the stats, and the hub paragraphs.


In [ ]:
# 3. Experiment — co-evidence graph over the sampled questions
def run_experiment() -> dict:
    questions = load_multi_hop(HOTPOT_PATH, N_QUESTIONS)

    graph = nx.Graph()
    for rec in questions:
        titles = {title for title, _ in rec["supporting_facts"]}
        for title in titles:
            graph.add_node(title, questions=[])
            graph.nodes[title]["questions"].append(rec["_id"])
        for a in titles:
            for b in titles:
                if a < b:
                    if not graph.has_edge(a, b):
                        graph.add_edge(a, b, weight=0)
                    graph[a][b]["weight"] += 1

    degrees = [d for _, d in graph.degree()]
    components = list(nx.connected_components(graph))
    chains = [(rec["question"], rec["_id"], evidence_chain(rec))
              for rec in questions]

    return {
        "questions": questions,
        "graph": graph,
        "chains": chains,
        "stats": {
            "nodes": graph.number_of_nodes(),
            "edges": graph.number_of_edges(),
            "max_degree": max(degrees) if degrees else 0,
            "components": len(components),
            "multi_hop_in_pool": len(questions),
        },
        "hubs": sorted(graph.degree(), key=lambda pair: pair[1],
                       reverse=True)[:TOP_HUBS],
    }


## 4. Demo

The demo prints the artifact in four parts: the graph's size and shape, the highest-degree paragraphs (shared evidence hubs), a few example evidence chains, and the takeaway. The interesting signal is the hub list — a paragraph that co-occurs with many different partners is one that many different questions reuse, which is exactly the structure GraphRAG's community detection (lab 03) later tries to rediscover from text.


In [ ]:
# 4. Demo — print the artifact
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-02 — HotpotQA evidence graph (co-evidence of gold paragraphs)")
    s = exp["stats"]
    print(f"{s['multi_hop_in_pool']} multi-hop questions in the pool")
    print("=" * 66)

    print(f"\n[1] Co-evidence graph: {s['nodes']} paragraphs, {s['edges']} "
          f"edges, {s['components']} components")
    print(f"    max paragraph degree: {s['max_degree']}")

    print(f"\n[2] Highest-degree paragraphs (shared evidence hubs):")
    for title, degree in exp["hubs"]:
        print(f"    {degree:3d}  {title[:70]}")

    print(f"\n[3] Example evidence chains (question -> gold paragraphs):")
    for question, qid, chain in exp["chains"][:MAX_CHAINS_SHOWN]:
        print(f"    Q: {question[:80]}")
        for title, sent_idx in chain:
            print(f"       ({title[:60]}, sentence {sent_idx})")

    print(f"\n[4] Takeaway")
    print("    An edge here is a question that *demands* both paragraphs.")
    print("    High-degree paragraphs are reuse hubs: one paragraph answers")
    print("    many different questions. This is the ground truth GraphRAG")
    print("    community detection (lab 03) tries to rediscover from text.")


## 5. Verification gate

The lab ships a `--verify` gate — the same gate `python src/curriculum/08-graphrag/02-evidence-graph.py --verify` runs — and the checks here are exactly those, applied to the notebook's `exp`: pool size, graph size, every edge weight a positive integer, every node remembering its questions, and every chain spanning >= 2 distinct paragraphs. The gate turns "the lab ran" into "the lab ran *correctly*" — and because every check is derived from the labels themselves, passing it also proves the graph is a faithful rendering of the data.


In [ ]:
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    s = exp["stats"]
    graph = exp["graph"]

    checks.append((f"pool has >= {N_QUESTIONS // 2} multi-hop questions "
                   f"(got {s['multi_hop_in_pool']})",
                   s["multi_hop_in_pool"] >= N_QUESTIONS // 2))
    checks.append((f"graph has >= 15 nodes (got {s['nodes']})",
                   s["nodes"] >= 15))
    checks.append((f"graph has >= 10 edges (got {s['edges']})",
                   s["edges"] >= 10))
    checks.append(("every edge weight is a positive integer",
                   all(graph[a][b]["weight"] >= 1 for a, b in graph.edges())))
    checks.append(("every node remembers its questions",
                   all(graph.nodes[n]["questions"] for n in graph.nodes())))
    checks.append(("every chain has >= 2 distinct paragraphs",
                   all(len({t for t, _ in chain}) >= 2
                       for _, _, chain in exp["chains"])))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Instant — there is no model to load, no embedding pass, no API call, no randomness. The only work is reading the first 50 multi-hop questions out of the dev set and building a small graph, so the cell completes in a second or two. `exp` holds everything the demo and gate need: the graph, the chains, the stats, and the hubs.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Four sections, straight from `exp`: graph stats, hub paragraphs, example evidence chains, and the takeaway. The chain lines are the money shot — each one is a question's gold evidence in the order a retriever would have to surface it.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
